# Create AMED (Japan Agency for Medical Research and Development) Awards

Creates OpenAlex award rows from AMED's own first-party project database, AMEDfind.

**Data source:** `https://amedfind.amed.go.jp/amed/` — AMEDfind, AMED's official R&D project DB (~11,055 projects). Records come from AMEDfind's undocumented search API `POST /amed/web/api/get/list` (requires the front-end header `x-request-origin: amedfind_web`; plain HTTP, no browser). Each result row carries the full project record, so no per-project detail fetch is needed. NOT KAKEN (KAKEN = MEXT/JSPS KAKENHI only and excludes AMED). The JST GRANTS portal (grants.jst.go.jp, qo=日本医療研究開発機構) federates AMEDfind but shows fewer records and paginates via JS only, so AMEDfind is authoritative.
**S3 location:** `s3a://openalex-ingest/awards/amed/amed_projects.parquet`
**OpenAlex funder:** `F4320311405` — Japan Agency for Medical Research and Development (ROR `https://ror.org/004rtk039`, DOI `10.13039/100009619`), country `JP`.
**Provenance:** `amed_amedfind`.
**Priority:** 207.

**Schema choices:**
- One row per AMED theme. Stable `funder_award_id = {tid}` (the theme-detail route id from the scraper); `theme_no` is the per-fiscal-year management number, kept for reference only.
- `amount` = `totalBudget` × 1000. AMEDfind publishes 課題への総配分額 in 千円 (thousands of yen); the scraper already converts to whole JPY, so the raw `amount` column is JPY. `currency = JPY`. Amounts are present (~100% in the smoke sample) — the §6.7 amount check is NOT waived (full-corpus coverage measured at ~93%).
- `funder_scheme` = `jigyo` (事業 / funding program, e.g. 革新的がん医療実用化研究事業).
- `funding_type` = `fellowship` if the program is a training/fellowship scheme (育成 / フェロー / 特別研究員), else `research`.
- `lead_investigator` = 研究代表者 (latest fiscal year). Japanese PI names are NOT space-delimited, so we do not guess a given/family split (runbook: NULL rather than guess): `family_name` = the verbatim name, `given_name` = NULL. `affiliation.name` = the PI institution (研究機関), `country = JP` (AMED requires a Japanese host institution).
- `start_date`/`end_date` come from the API's `allStartDate`/`allEndDate` (full ISO dates), and `start_year`/`end_year` are derived from them.

**Prerequisite:** run `scripts/local/amed_to_s3.py` to populate the S3 parquet. The scraper pages through the `get/list` API with plain HTTP (the only non-obvious requirement is the front-end header `x-request-origin: amedfind_web`; no browser needed). See the script docstring.

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.amed_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/amed/amed_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) AS total_amed_projects FROM openalex.awards.amed_raw;

## Step 1.5: Inspect Raw Data, Money Fields, and Key Uniqueness

Expected source columns (from `amed_to_s3.py`): `funder_award_id` (entrusterThemeNo), `theme_no`, `theme_id`, `display_name`, `description`, `funder_scheme`, `integrated_project`, `amount` (whole JPY), `amount_thousand_yen` (raw 千円), `currency`, `lead_researcher_name`, `lead_affiliation_name`, `lead_job`, `researcher_id`, `erad_researcher_id`, `start_date`, `end_date`, `start_year`, `end_year`, `landing_page_url`, `downloaded_at`.

`amount` is already in whole JPY (the scraper multiplied the 千円 `totalBudget` by 1000). Verify the distribution lands in the millions/billions, not the thousands. Amount coverage is ~93% (newest current-fiscal-year projects have totalBudget 0 until allocations are reported).

In [ ]:
%sql
DESCRIBE openalex.awards.amed_raw;

In [ ]:
%sql
SELECT funder_award_id, display_name, funder_scheme, lead_researcher_name,
       lead_affiliation_name, amount, amount_thousand_yen, currency,
       start_date, end_date, start_year, end_year, landing_page_url
FROM openalex.awards.amed_raw
LIMIT 10;

In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(funder_scheme) AS has_scheme,
    COUNT(lead_researcher_name) AS has_pi,
    COUNT(lead_affiliation_name) AS has_affiliation,
    COUNT(TRY_CAST(amount AS DOUBLE)) AS has_amount,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    ROUND(AVG(TRY_CAST(amount AS DOUBLE)), 0) AS avg_amount,
    COUNT(TRY_CAST(start_year AS INT)) AS has_start_year,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_start_year,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    ROUND(try_divide(COUNT(TRY_CAST(amount AS DOUBLE)), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.amed_raw;

## Step 1.6: Fail-Fast - Verify AMED Funder Exists

F4320311405 is a Crossref-registered `F4320*` funder (Path A), so it should be present in `openalex.common.funder`. The transform CROSS JOINs to a `funder_resolved` CTE that reads the dim with a hardcoded fallback to the canonical values, so the awards never silently drop to zero if the dim is missing the row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320311405;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.amed_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT
        4320311405 AS funder_id,
        COALESCE(MAX(display_name), 'Japan Agency for Medical Research and Development') AS display_name,
        COALESCE(MAX(ror_id), 'https://ror.org/004rtk039') AS ror_id,
        COALESCE(MAX(doi), '10.13039/100009619') AS doi
    FROM openalex.common.funder
    WHERE funder_id = 4320311405
),
raw_prepped AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS amount_double,
        CASE WHEN TRY_CAST(start_year AS INT) BETWEEN 1800 AND 2100 THEN TRY_CAST(start_year AS INT) END AS start_year_int,
        CASE WHEN TRY_CAST(end_year AS INT) BETWEEN 1800 AND 2100 THEN TRY_CAST(end_year AS INT) END AS end_year_int,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS start_date_parsed,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS end_date_parsed,
        NULLIF(TRIM(lead_affiliation_name), '') AS leader_affiliation_name,
        NULLIF(TRIM(lead_researcher_name), '') AS leader_name
    FROM openalex.awards.amed_raw
)
SELECT
    abs(xxhash64(CONCAT('4320311405:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS id,
    rp.display_name,
    NULLIF(TRIM(rp.description), '') AS description,
    4320311405 AS funder_id,
    rp.funder_award_id,
    rp.amount_double AS amount,
    CASE WHEN rp.amount_double IS NOT NULL THEN 'JPY' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    CASE
        WHEN rp.funder_scheme RLIKE '育成|フェロー|特別研究員' THEN 'fellowship'
        ELSE 'research'
    END AS funding_type,
    NULLIF(TRIM(rp.funder_scheme), '') AS funder_scheme,
    'amed_amedfind' AS provenance,
    rp.start_date_parsed AS start_date,
    rp.end_date_parsed AS end_date,
    rp.start_year_int AS start_year,
    rp.end_year_int AS end_year,
    CASE
        WHEN rp.leader_name IS NOT NULL OR rp.leader_affiliation_name IS NOT NULL THEN
            struct(
                CAST(NULL AS STRING) AS given_name,
                rp.leader_name AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    rp.leader_affiliation_name AS name,
                    CASE WHEN rp.leader_affiliation_name IS NOT NULL THEN 'JP' ELSE NULL END AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    rp.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT('4320311405:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM raw_prepped rp
CROSS JOIN funder_resolved f
WHERE rp.funder_award_id IS NOT NULL
  AND rp.display_name IS NOT NULL;

## Step 3: Insert into `openalex_awards_raw` at Priority 207

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'amed_amedfind' AND priority = 207;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    207 AS priority
FROM openalex.awards.amed_awards;

## Step 6: Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_amed_awards FROM openalex.awards.amed_awards;

In [ ]:
%sql
-- Duplicate funder_award_id / id must be zero.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids
FROM openalex.awards.amed_awards;

In [ ]:
%sql
-- Completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_positive_amount,
    SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_name,
    SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_affiliation,
    COUNT(start_year) AS has_start_year,
    COUNT(funder_scheme) AS has_funder_scheme,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_lead_name,
    collect_set(currency) AS currencies
FROM openalex.awards.amed_awards;

In [ ]:
%sql
-- Amount coverage (NOT waived: AMEDfind publishes 課題への総配分額). Must be >50% per §6.7.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount, MAX(amount) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount, percentile_approx(amount, 0.5) AS median_amount
FROM openalex.awards.amed_awards;

In [ ]:
%sql
-- PI frequency (Step 6.4a): no name should dominate; no institution-as-name.
SELECT lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.amed_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT funding_type, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS jpy_total
FROM openalex.awards.amed_awards
GROUP BY funding_type ORDER BY awards DESC;

In [ ]:
%sql
-- Top programs (jigyo) by award count.
SELECT funder_scheme, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS jpy_total
FROM openalex.awards.amed_awards
GROUP BY funder_scheme ORDER BY awards DESC LIMIT 20;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 60) AS title, amount, currency, start_year, end_year,
       lead_investigator.family_name AS pi,
       SUBSTRING(lead_investigator.affiliation.name, 1, 40) AS affiliation
FROM openalex.awards.amed_awards
ORDER BY start_year DESC, id LIMIT 20;

In [ ]:
%sql
-- Confirm the priority insert landed.
SELECT priority, provenance, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'amed_amedfind'
GROUP BY priority, provenance;